# FireScore end-to-end — Sevier County, TN

This notebook walks the full workflow on the real Sevier County project
built on **2026-11-05**. On a fresh machine the two commands below build
everything from free national data (about 35 minutes, most of it waiting
on the LANDFIRE job queue; every later step is offline):

```bash
firescore fetch --county "Sevier, TN" --project ~/firescore-projects/sevier_47155
firescore run   --county "Sevier, TN" --project ~/firescore-projects/sevier_47155 --offline
```

Recorded fetch result: 75,227 FEMA USA Structures footprints; OSM road
graph 9,497 nodes / 21,614 edges; 11 OSM stations; **0 OSM hydrants**
(water resolved to the labelled draft-site proxy, 9,739 water features);
USGS 3DEP DEM; LANDFIRE LF2023 FBFM40; USFS WRC burn probability;
14 validation perimeters incl. Chimney Tops 2 (2016) and Indigo Lane /
Hatcher Mountain (2022).
Recorded run result: **75,227 structures scored offline in ~90 s**.

The cells below work against that built project — no network is used.

In [1]:
import json
from pathlib import Path
import geopandas as gpd
import pandas as pd

PROJECT = Path.home() / "workspace/firescore/projects/sevier_47155"
manifest = json.loads((PROJECT / "cache/manifest.json").read_text())
pd.Series({k: v.get("rows") for k, v in manifest["layers"].items()}).rename("cached rows")

boundary           1.0
footprints     75227.0
roads_nodes     9497.0
roads_edges    21614.0
stations          11.0
dem                NaN
fbfm40             NaN
hydrants           0.0
draft_sites     9739.0
wrc_bp             NaN
perimeters        14.0
Name: cached rows, dtype: float64

## The scored layer

Every score is a **range** across five weight configurations (`score_low`–`score_high`), never a bare point value.

In [2]:
scored = gpd.read_parquet(PROJECT / "outputs/scores.parquet")
summary = json.loads((PROJECT / "outputs/score_summary.json").read_text())
print(f"{len(scored):,} structures | layer role: {summary['layer_role']} | "
      f"water source: {summary['water_source']}")
scored["vuln_class"].value_counts().reindex(["Low", "Moderate", "High", "Extreme"])

75,227 structures | layer role: footprint_proxy | water source: osm_draft_sites_PROXY


vuln_class
Low         15694
Moderate    49394
High         9984
Extreme       155
Name: count, dtype: int64

In [3]:
comp = [c for c in scored.columns if c.startswith("comp_")]
scored[comp].mean().round(3).rename("county mean component")

comp_egress        0.396
comp_water         0.532
comp_vegetation    0.457
comp_terrain       0.291
comp_spacing       0.492
Name: county mean component, dtype: float64

## The worst of the queue

The inspection queue ranks by the upper bound of the range, so structures that are worst under *any* defensible weighting come first.

In [4]:
queue = pd.read_csv(PROJECT / "outputs/inspection_queue.csv")
# Published examples show aggregate geography only (community level) -
# never street addresses. The address-level queue is a local file.
queue.head(5)[["queue_rank", "PROP_CITY", "score", "score_low", "score_high", "vuln_class", "cluster_exits"]]

,queue_rank,PROP_CITY,score,score_low,score_high,vuln_class,cluster_exits
0,1,SEYMOUR,0.843141,0.772734,0.915246,Extreme,0
1,2,SEYMOUR,0.842838,0.772571,0.915084,Extreme,0
2,3,SEYMOUR,0.825257,0.783800,0.905070,Extreme,0
3,4,SEYMOUR,0.824510,0.783400,0.904670,Extreme,0
4,5,SEVIERVILLE,0.815464,0.762850,0.901116,Extreme,1


## What-if: rank mitigations for the worst structure

Actions are ranked by score drop per $1,000 at planning-level costs from the versioned config. A hydrant can only *improve* a structure's water term — the editor never raises a score.

In [5]:
from firescore.cache import CountyCache
from firescore.config import load_config
from firescore.whatif import WhatIfSession

cfg = load_config(None)
session = WhatIfSession(CountyCache(PROJECT), cfg, scored)
worst_id = queue.iloc[0]["structure_id"]  # local id only; no address is printed
ranked = session.rank_actions(worst_id)
pd.DataFrame(ranked)[["action", "score_before", "score_after", "delta", "cost_usd", "drop_per_1000_usd", "structures_improved"]]

,action,score_before,score_after,delta,cost_usd,drop_per_1000_usd,structures_improved
0,clear_veg,0.843141,0.772581,-0.07056,3000.00000,0.023520,2
1,add_hydrant,0.843141,0.703141,-0.14000,6500.00000,0.021538,14
2,add_road,0.843141,0.808141,-0.03500,141610.17267,0.000247,32


## Validation diagnostics

Outcome = structure inside the fire perimeter (not observed loss). Read the `_outcome_note` and any per-fire `diagnostic` before citing a number: on Chimney Tops 2 the composite is *inverted*, because that perimeter is mostly uninhabited park interior — the module flags this itself.

In [6]:
val = json.loads((PROJECT / "outputs/validation.json").read_text())
for name in ("Chimney Tops 2", "INDIGO LANE"):
    r = val[name]
    print(name, "| inside:", r["inside_perimeter"],
          "| composite AUC:", round(r["composite"]["auc"], 3),
          "| RF block-CV:", round(r["rf_block_cv_auc_mean"], 3),
          "| WRC AUC:", round(r["wrc"]["auc"], 3),
          "| diagnostic:", "diagnostic" in r)
print()
print("FireScore vs WRC burn probability, Spearman rho:",
      round(val["Chimney Tops 2"]["wrc"]["spearman_vs_firescore"], 3))

Chimney Tops 2 | inside: 2873 | composite AUC: 0.229 | RF block-CV: 0.925 | WRC AUC: 0.379 | diagnostic: True
INDIGO LANE | inside: 36 | composite AUC: 0.474 | RF block-CV: 0.49 | WRC AUC: 0.921 | diagnostic: False

FireScore vs WRC burn probability, Spearman rho: 0.226


## Take the layer into the field

```bash
firescore export --county "Sevier, TN" --project <project>   # GPKG / SHP / GeoJSON+CSV / FireView
firescore report --county "Sevier, TN" --project <project> --top 25   # one-page PDF cards
```

FireView is a self-contained folder (vendored Leaflet, local data, zero
external references) that opens from `file://` on a phone with no signal.
The second-county proof: Blount County, TN ran on the same code and
default config, unmodified — 79,056 structures, offline.